# IR_PROJECT: GPU experiments and evaluation

This notebook is for **Google Colab or Kaggle with a GPU runtime**. Do not run training, full-corpus evaluation, or query-enhancement cells on your laptop. Run the cells in order, review each result, and keep the generated bundle for later review.

It prepares separate checkpoints and output folders so the restored legacy models/results are not overwritten. It runs the requested experiments: longer hard-negative and random-negative bi-encoders; a query-preserving, NLI-initialized cross-encoder; TF-IDF/BM25/neural/hybrid comparisons with paired confidence intervals; candidate/rerank-depth and query-length/negative-sampling ablations; BM25 and TF-IDF query enhancement; and optional zero-shot BEIR SciFact.

**Important protocol notes**
- The legacy cross-encoder was initialized from an MS MARCO model and its classification head was discarded. Reusing an MS MARCO-trained scoring head could contaminate an MS MARCO test comparison. This notebook uses the NLI checkpoint configured in the project, preserves its trained classifier head, and converts NLI logits to an entailment-minus-contradiction relevance score. It still must be fine-tuned and validated; no improvement is assumed.
- Final test results are descriptive, not a tuning target. Pick checkpoints using validation; run the test comparison once after choices are fixed.
- The Hugging Face Space is a separate CPU-served corpus/checkpoint setup. Offline Colab numbers are not live Space measurements.
- Query-enhancement previously failed due an unbounded output-path list; that bug is fixed. This notebook runs it against compact single-retriever indexes.

Notebook outputs are not embedded. The notebook file itself is stored outside the Git repositories; experiment artifacts are created in the runtime and should be saved to Drive or downloaded.


In [1]:
# Platform setup: choose GPU in Runtime settings (Colab) or Settings (Kaggle) first.
from pathlib import Path
import os
import subprocess
import sys

IS_COLAB = 'google.colab' in sys.modules
IS_KAGGLE = Path('/kaggle/working').exists()
if not (IS_COLAB or IS_KAGGLE):
    raise RuntimeError('Open this notebook in Google Colab or Kaggle, not on the local laptop.')

WORK_ROOT = Path('/content') if IS_COLAB else Path('/kaggle/working')
GIT_BRANCH = 'Dhiraj'  # Change if you pushed the project updates to another branch.
REPO_DIR = WORK_ROOT / 'IR_PROJECT'
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', '--branch', GIT_BRANCH, 'https://github.com/Dhiraj7854/IR_PROJECT.git', str(REPO_DIR)], check=True)
PROJECT_DIR = REPO_DIR / 'neural-ranking'
if not PROJECT_DIR.is_dir():
    raise FileNotFoundError(f'Expected project directory: {PROJECT_DIR}')
os.chdir(PROJECT_DIR)
print('Platform:', 'Colab' if IS_COLAB else 'Kaggle')
print('Project:', PROJECT_DIR)
print('Working directory:', Path.cwd())


Cloning into '/kaggle/working/IR_PROJECT'...


Platform: Kaggle
Project: /kaggle/working/IR_PROJECT/neural-ranking
Working directory: /kaggle/working/IR_PROJECT/neural-ranking


## 1. Install the one missing retrieval package

Kaggle already provides CUDA-enabled PyTorch, Transformers, Sentence-Transformers, pandas, and scikit-learn. Keep those preinstalled versions: pinning this project's older Transformers/Sentence-Transformers versions downgraded `huggingface-hub` and produced conflicts with Kaggle's Gradio and Diffusers packages. The project scripts add their own folder to `sys.path`, so `pip install -e .` is also unnecessary. Only `rank-bm25` was missing in the attached Kaggle install output, so install that package alone and leave Kaggle's model/CUDA stack untouched.


In [2]:
%pip install "rank-bm25==0.2.2"


Note: you may need to restart the kernel to use updated packages.


## 2. Confirm a GPU is available

If this cell reports no CUDA device, stop and enable a GPU accelerator in the notebook settings before proceeding.


In [3]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('GPU required: enable a CUDA GPU runtime, then restart and rerun the setup cells.')
print('GPU:', torch.cuda.get_device_name(0))
print('GPU memory (GiB):', round(torch.cuda.get_device_properties(0).total_memory / 2**30, 2))


PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4
GPU memory (GiB): 14.56


## 3. Connect prepared data (do not silently redownload it)

The processed MS MARCO files are usually not committed to Git. Set `SOURCE_DATA_DIR` to the prepared `data/processed` directory from your Drive bundle (Colab) or attached Kaggle dataset. The directory must contain `corpus.parquet`, train/validation/test query and qrels files, and `train_triplets.parquet`.

For Colab, the optional Drive mount below is explicit. For Kaggle, add the dataset through the notebook's Input panel and set the path. If your files are inside an archive, extract them through the Kaggle/Colab UI or a separate reviewed step, then point `SOURCE_DATA_DIR` at `data/processed`. `prepare_data.py` is provided as an opt-in fallback but may download a large dataset.


In [4]:
if IS_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

required = [
    'corpus.parquet', 'train_queries.parquet', 'train_qrels.parquet',
    'train_triplets.parquet', 'val_queries.parquet', 'val_qrels.parquet',
    'test_queries.parquet', 'test_qrels.parquet',
]
DATA_DIR = PROJECT_DIR / 'data' / 'processed'

# Kaggle mounts datasets read-only under /kaggle/input/<dataset-name>/.
# Search for the prepared files so the dataset slug does not need guessing.
if IS_KAGGLE:
    input_root = Path('/kaggle/input')
    candidates = [path.parent for path in input_root.rglob('corpus.parquet')]
    complete_candidates = [
        path for path in candidates
        if all((path / filename).is_file() for filename in required)
    ]
    if len(complete_candidates) == 1:
        SOURCE_DATA_DIR = complete_candidates[0]
    elif len(complete_candidates) > 1:
        print('Found multiple prepared data folders:')
        for candidate in complete_candidates:
            print(' -', candidate)
        raise RuntimeError(
            'Set SOURCE_DATA_DIR explicitly to the folder containing the matching prepared dataset.'
        )
    else:
        inputs = sorted(str(path) for path in input_root.iterdir()) if input_root.exists() else []
        raise FileNotFoundError(
            'Prepared MS MARCO files were not found under /kaggle/input. '
            'In Kaggle, use Add Input to attach a Kaggle Dataset containing '
            'data/processed/*.parquet (the eight required files listed above). '
            'If your files are in a ZIP, first add a Kaggle Dataset with the '
            'extracted data/processed folder, or extract the ZIP into /kaggle/working '
            'and set SOURCE_DATA_DIR to that extracted folder. '
            f'Current Kaggle inputs: {inputs}'
        )
else:
    # Colab path to the prepared data restored in Drive; edit if your folder differs.
    SOURCE_DATA_DIR = Path('/content/drive/MyDrive/IR_PROJECT/neural-ranking/data/processed')
    if not SOURCE_DATA_DIR.is_dir():
        raise FileNotFoundError(
            f'Prepared data folder not found: {SOURCE_DATA_DIR}. '
            'Mount the Drive folder containing data/processed, then update SOURCE_DATA_DIR.'
        )

missing = [name for name in required if not (SOURCE_DATA_DIR / name).is_file()]
if missing:
    raise FileNotFoundError(
        f'{SOURCE_DATA_DIR} is not a complete prepared data/processed folder; missing: {missing}'
    )
if DATA_DIR.exists():
    missing_local = [name for name in required if not (DATA_DIR / name).is_file()]
    if missing_local:
        raise FileExistsError(f'Existing data directory is incomplete: {DATA_DIR}; missing {missing_local}')
    print('Reusing existing prepared data directory.')
elif SOURCE_DATA_DIR.resolve() != DATA_DIR.resolve():
    import shutil
    DATA_DIR.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(SOURCE_DATA_DIR, DATA_DIR)
print('Prepared data:', DATA_DIR)
print('Source:', SOURCE_DATA_DIR)
print('Files:', len(list(DATA_DIR.glob('*'))))


Prepared data: /kaggle/working/IR_PROJECT/neural-ranking/data/processed
Source: /kaggle/input/datasets/dhirajkumardubey/ms-marco
Files: 8


## 4. Create a fresh, isolated experiment output directory

Every checkpoint and generated run goes under a timestamped folder in `artifacts/colab_runs/`. Existing `artifacts/biencoder`, `artifacts/crossencoder`, and restored result files are not used as output targets. Do not change these paths to the old checkpoint directories.


In [5]:
from datetime import datetime, timezone

RUN_ID = datetime.now(timezone.utc).strftime('run_%Y%m%dT%H%M%SZ')
ARTIFACTS_DIR = PROJECT_DIR / 'artifacts' / 'colab_runs' / RUN_ID
if ARTIFACTS_DIR.exists():
    raise FileExistsError(f'Refusing to reuse an experiment folder: {ARTIFACTS_DIR}')
ARTIFACTS_DIR.mkdir(parents=True)
BI_HARD = ARTIFACTS_DIR / 'biencoder_hard_extended'
BI_RANDOM = ARTIFACTS_DIR / 'biencoder_random'
CROSS_NLI = ARTIFACTS_DIR / 'crossencoder_nli'
RUNS_DIR = ARTIFACTS_DIR / 'runs'
RESULTS_DIR = ARTIFACTS_DIR / 'results'
print('Run ID:', RUN_ID)
print('Artifacts:', ARTIFACTS_DIR)
print('Disk free (GiB):', round(__import__('shutil').disk_usage(ARTIFACTS_DIR).free / 2**30, 2))


Run ID: run_20261023T090625Z
Artifacts: /kaggle/working/IR_PROJECT/neural-ranking/artifacts/colab_runs/run_20261023T090625Z
Disk free (GiB): 19.46


## 5. Train the bi-encoder with hard and random negatives

The prior bundle recorded only one epoch/55 optimizer steps for its hard-negative model, and it had no hard-vs-random result. These two runs use the same model, seed, epochs, and validation sample size; only the negative-sampling mode changes. Check the saved `training_metrics.json` for completed epochs, global steps, and validation history. The 2,000-passage validation sample is a checkpoint-selection diagnostic; it is not comparable with full-corpus test metrics.


In [6]:
def run_project(args):
    command = [sys.executable, *map(str, args)]
    print('RUN:', ' '.join(command))
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

run_project([
    'scripts/train_biencoder.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--output-dir', BI_HARD, '--negatives', 'hard',
    '--epochs', '5', '--patience', '3', '--val-sample-size', '2000', '--seed', '42',
])


RUN: /usr/bin/python3 scripts/train_biencoder.py --config configs/config.yaml --data-dir /kaggle/working/IR_PROJECT/neural-ranking/data/processed --output-dir /kaggle/working/IR_PROJECT/neural-ranking/artifacts/colab_runs/run_20261023T090625Z/biencoder_hard_extended --negatives hard --epochs 5 --patience 3 --val-sample-size 2000 --seed 42


2026-10-23 09:06:49,812 INFO TensorFlow version 2.20.0 available.
2026-10-23 09:06:49,814 INFO JAX version 0.11.1 available.
/kaggle/working/IR_PROJECT/neural-ranking/scripts/train_biencoder.py:278: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses as st_losses
2026-10-23 09:06:54,646 INFO HTTP Request: GET https://huggingface.co/api/agent-harnesses "HTTP/1.1 200 OK"
2026-10-23 09:06:54,663 INFO HTTP Request: HEAD https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"
2026-10-23 09:06:54,663 WARNING Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-10-23 09:06:54,669 INFO HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/sentence

In [7]:
run_project([
    'scripts/train_biencoder.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--output-dir', BI_RANDOM, '--negatives', 'random',
    '--epochs', '5', '--patience', '3', '--val-sample-size', '2000', '--seed', '42',
])


RUN: /usr/bin/python3 scripts/train_biencoder.py --config configs/config.yaml --data-dir /kaggle/working/IR_PROJECT/neural-ranking/data/processed --output-dir /kaggle/working/IR_PROJECT/neural-ranking/artifacts/colab_runs/run_20261023T090625Z/biencoder_random --negatives random --epochs 5 --patience 3 --val-sample-size 2000 --seed 42


2026-10-23 09:08:14,561 INFO TensorFlow version 2.20.0 available.
2026-10-23 09:08:14,562 INFO JAX version 0.11.1 available.
/kaggle/working/IR_PROJECT/neural-ranking/scripts/train_biencoder.py:278: DeprecationWarning: Importing from 'sentence_transformers.losses' is deprecated and will be removed in a future version. Please use 'sentence_transformers.sentence_transformer.losses' instead.
  from sentence_transformers import losses as st_losses
Traceback (most recent call last):
  File "/kaggle/working/IR_PROJECT/neural-ranking/scripts/train_biencoder.py", line 479, in <module>
    main()
    ~~~~^^
  File "/kaggle/working/IR_PROJECT/neural-ranking/scripts/train_biencoder.py", line 336, in main
    examples = _create_training_examples(
        triplets_df,
    ...<2 lines>...
        seed=args.seed,
    )
  File "/kaggle/working/IR_PROJECT/neural-ranking/scripts/train_biencoder.py", line 201, in _create_training_examples
    random_negatives = sample_random_negative_texts(
        corpu

CalledProcessError: Command '['/usr/bin/python3', 'scripts/train_biencoder.py', '--config', 'configs/config.yaml', '--data-dir', '/kaggle/working/IR_PROJECT/neural-ranking/data/processed', '--output-dir', '/kaggle/working/IR_PROJECT/neural-ranking/artifacts/colab_runs/run_20261023T090625Z/biencoder_random', '--negatives', 'random', '--epochs', '5', '--patience', '3', '--val-sample-size', '2000', '--seed', '42']' returned non-zero exit status 1.

## 6. Train the NLI-initialized cross-encoder

The cross-encoder keeps the NLI-trained sequence-classification head and scores query-passage pairs by entailment-logit minus contradiction-logit. Pair tokenization uses `truncation='only_second'`, preserving the query and truncating passage text if needed. This is a leakage-conscious alternative to using an MS MARCO-trained scoring head for an MS MARCO test evaluation. It does not guarantee better ranking; inspect the validation history and candidate-depth experiments.


In [ ]:
run_project([
    'scripts/train_crossencoder.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--model-name', 'cross-encoder/nli-MiniLM2-L6-H768',
    '--output-dir', CROSS_NLI, '--epochs', '3', '--batch-size', '32',
    '--max-length', '256', '--seed', '42',
])


## 7. Produce TF-IDF, BM25, neural, hybrid, and reranked runs

These commands evaluate validation and test splits. Keep test results for final reporting only; do not tune hyperparameters from the test output. The paired bootstrap file created later reports differences against both BM25 and TF-IDF.


In [ ]:
run_project([
    'scripts/run_baseline.py', '--artifacts_dir', ARTIFACTS_DIR,
    '--data_dir', DATA_DIR, '--splits', 'val', 'test',
    '--methods', 'tfidf', 'bm25', '--top_k', '100',
])
run_project([
    'scripts/eval_biencoder.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
    '--model', BI_HARD, '--splits', 'val', 'test', '--top-k', '100',
])
run_project([
    'scripts/eval_crossencoder.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
    '--model', CROSS_NLI, '--candidate-depth', '100', '--top-k', '100',
    '--splits', 'val', 'test',
])
run_project([
    'scripts/eval_hybrid.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
    '--biencoder-model', BI_HARD, '--crossencoder-model', CROSS_NLI,
    '--candidate-depth', '100', '--rerank-depth', '50', '--top-k', '100',
    '--splits', 'val', 'test',
])


## 8. Cross-encoder candidate/rerank-depth diagnosis

Compare the same NLI-initialized ranker against BM25 and RRF candidates. The first line specifically checks the shallow BM25 top-10 setting that had the best legacy validation MRR. Outputs are depth-specific. The 100/50 line is also used in the main comparison. This experiment helps separate poor candidate coverage from reranking damage; do not infer a cross-encoder limitation from one checkpoint.


In [ ]:
for candidate_depth, rerank_depth in [(10, 10), (50, 50), (100, 50)]:
    run_project([
        'scripts/eval_hybrid.py', '--config', 'configs/config.yaml',
        '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
        '--biencoder-model', BI_HARD, '--crossencoder-model', CROSS_NLI,
        '--candidate-depth', str(candidate_depth), '--rerank-depth', str(rerank_depth),
        '--top-k', '100', '--splits', 'val', 'test',
    ])


## 9. Hard-vs-random, rerank-depth, and query-length ablations

This evaluates hard/random bi-encoders on validation, BM25 rerank depths, and test-query word-length terciles. The length analysis is descriptive on the test split, not a tuning procedure. This command writes an ablation report and plots in `results/`.


In [ ]:
run_project([
    'scripts/run_experiments.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
    '--biencoder-model', BI_HARD, '--crossencoder-model', CROSS_NLI,
    '--reuse-random-checkpoint', '--random-biencoder-model', BI_RANDOM, '--seed', '42',
])


## 10. Main table and paired confidence intervals vs BM25 and TF-IDF

This command measures latency on 200 test queries. Latencies depend on this cloud GPU/CPU setup and must not be compared directly with the Hugging Face Space's CPU-served live latency. The output `paired_differences.json` keeps the legacy BM25 fields and adds a nested TF-IDF comparison for each system.


In [ ]:
run_project([
    'scripts/run_all_eval.py', '--config', 'configs/config.yaml',
    '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
    '--biencoder-model', BI_HARD, '--crossencoder-model', CROSS_NLI,
    '--latency-queries', '200', '--seed', '42',
])


In [ ]:
import json
paired_path = RESULTS_DIR / 'paired_differences.json'
paired = json.loads(paired_path.read_text(encoding='utf-8'))
for system, comparison in paired.items():
    tfidf = comparison.get('additional_comparisons', {}).get('TF-IDF', {}).get('comparison_minus_baseline', {}).get('MRR@10')
    bm25 = comparison.get('comparison_minus_baseline', {}).get('MRR@10')
    print(system)
    if bm25:
        print('  vs BM25 :', f"{bm25['mean_difference']:+.4f}", f"95% CI [{bm25['ci_lower']:+.4f}, {bm25['ci_upper']:+.4f}]")
    if tfidf:
        print('  vs TF-IDF:', f"{tfidf['mean_difference']:+.4f}", f"95% CI [{tfidf['ci_lower']:+.4f}, {tfidf['ci_upper']:+.4f}]")
assert all('TF-IDF' in item.get('additional_comparisons', {}) for name, item in paired.items() if name != 'TF-IDF')
print('Paired comparisons are present for both required lexical baselines.')


## 11. Query enhancement: BM25 and TF-IDF

The fixed implementation constructs only the selected lexical index and uses a finite output-path list. It does not build an unused second index. Run normalization and synonym expansion on both retrievers; spelling correction is optional. These experiments are separate from the main benchmark and must be reported with their own paired differences and exact run artifacts.


In [ ]:
for retriever in ('bm25', 'tfidf'):
    run_project([
        'scripts/eval_query_enhancement.py', '--config', 'configs/config.yaml',
        '--data-dir', DATA_DIR, '--artifacts-dir', ARTIFACTS_DIR,
        '--retriever', retriever, '--methods', 'original', 'normalize', 'synonym',
        '--splits', 'val', 'test',
    ])


## 12. Optional BEIR SciFact zero-shot generalization

Enable this section only after the in-domain experiments have completed and there is enough runtime/disk. It explicitly installs the optional `beir` package and downloads the public SciFact archive. Evaluation uses the dataset's test split without tuning on its qrels. The project bi-encoder and NLI cross-encoder are transferred without SciFact fine-tuning; results therefore measure zero-shot transfer of these project checkpoints.


In [ ]:
RUN_SCIFACT = True  # Set False to skip the download and BEIR evaluation.
if RUN_SCIFACT:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'beir==2.0.0'], check=True)
    run_project(['scripts/prepare_beir.py', '--dataset', 'scifact', '--data-dir', PROJECT_DIR / 'data' / 'beir'])
    run_project([
        'scripts/eval_beir.py', '--dataset', 'scifact',
        '--data-dir', PROJECT_DIR / 'data' / 'beir', '--artifacts-dir', ARTIFACTS_DIR,
        '--biencoder-model', BI_HARD, '--crossencoder-model', CROSS_NLI,
        '--candidate-depth', '100', '--rerank-depth', '50', '--top-k', '100',
    ])
else:
    print('Skipped optional SciFact download/evaluation.')


## 13. Review artifacts and package the run

Do not write new report claims from console output alone. Review the comparison CSV/Markdown, paired intervals, training histories, ablation report, query-enhancement metric files, BEIR metrics, query/qrels hashes, and evaluation manifests. Preserve all generated files so someone else can reproduce the tables.

The bundle includes checkpoints and can be large. The following cell creates a zip in the cloud working directory. In Colab it also offers a browser download. For persistence, copy the zip to mounted Drive. In Kaggle, use the Output panel to download `/kaggle/working/` results.


In [ ]:
import shutil
import zipfile

archive_path = WORK_ROOT / f'{RUN_ID}.zip'
if archive_path.exists():
    raise FileExistsError(f'Refusing to overwrite archive: {archive_path}')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=1) as archive:
    for file_path in ARTIFACTS_DIR.rglob('*'):
        if file_path.is_file():
            archive.write(file_path, file_path.relative_to(ARTIFACTS_DIR.parent))
print('Created:', archive_path)
print('Size (MiB):', round(archive_path.stat().st_size / 2**20, 1))

if IS_COLAB:
    from google.colab import files
    files.download(str(archive_path))
    # Optional persistent Drive copy; change this destination if needed.
    DRIVE_OUTPUT = Path('/content/drive/MyDrive/IR_PROJECT/colab-results')
    if Path('/content/drive').is_mount():
        DRIVE_OUTPUT.mkdir(parents=True, exist_ok=True)
        drive_copy = DRIVE_OUTPUT / archive_path.name
        shutil.copy2(archive_path, drive_copy)
        print('Drive copy:', drive_copy)


## Presentation wording and result checklist

State that the benchmark table is an offline evaluation on the recorded MS MARCO corpus/checkpoints, while the deployed Space uses a separate packaged corpus and models and runs neural inference on CPU by default. The offline table is not a measurement of live Space latency or quality. Make a short backup recording of the working Space before presenting; the notebook cannot create a recording of a live deployment.

Before making claims, verify: (1) both model training histories and actual steps/epochs; (2) all runs use matching qrels/query IDs; (3) the evaluation manifest points to the intended new checkpoints; (4) confidence intervals include BM25 and TF-IDF; (5) SciFact reports its dataset and zero-shot mode; (6) ablation artifacts exist; and (7) the notebook run bundle is saved. Missing or failed experiments should remain labeled pending, not be filled with estimated values.
